# parameter recovery

colors trial type:


yellow = 1

orange = 2

red = 3

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["OMP_NUM_THREADS"] = "1"
import seaborn as sns 
from scipy.io import loadmat
import ast
from scipy.stats import pearsonr
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import FixedLocator
from scipy.spatial.distance import euclidean
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from scipy.io import loadmat, savemat
import warnings
warnings.filterwarnings("ignore")

# read mean and std of red, orange, and yellow reward summary

In [ ]:
folder = "param_recovery_2_reward_distribution"
file_path = os.path.join(folder, "reward_summary_by_color.csv")

reward_summary = pd.read_csv(file_path)

color_to_num = {
    "yellow": 1,
    "orange": 2,
    "red": 3
}

reward_stats = {
    color_to_num[row["color"]]: {
        "mean": row["mean_reward"],
        "std": row["std_reward"]
    }
    for _, row in reward_summary.iterrows()
}

# examples:
# reward_stats[3]["mean"]   # red mean
# reward_stats[3]["std"]    # red std
# reward_stats[2]["mean"]   # orange mean
# reward_stats[1]["mean"]   # yellow mean

In [ ]:
outputFolderName = r"\\155.100.91.44\d\Data\Nill\BART_param_recovery\new_modeling\param_recovery_2_simulated_fields"
inputFolderName = r"\\155.100.91.44\d\Data\Nill\BART_param_recovery\new_modeling\param_recovery_1_data"

if not os.path.exists(outputFolderName):
    os.makedirs(outputFolderName)
    

matFiles = [f for f in os.listdir(inputFolderName) if f.endswith(".mat")]
nPatients = len(matFiles)

In [ ]:
def safe_scalar(x):
    """Convert matlab-loaded scalar/0d/1-element array to python scalar."""
    arr = np.asarray(x).squeeze()
    if arr.shape == ():
        return arr.item()
    return arr

def sample_truncated_normal(mean_val, std_val, low=None, high=None, max_tries=10000):
    """
    Sample from a normal distribution with optional lower/upper bounds.
    Falls back safely if repeated draws fail.
    """
    if std_val <= 0 or np.isnan(std_val):
        val = mean_val
        if low is not None:
            val = max(val, low)
        if high is not None:
            val = min(val, high)
        return float(max(0, val))

    for _ in range(max_tries):
        val = np.random.normal(mean_val, std_val)
        if low is not None and val < low:
            continue
        if high is not None and val > high:
            continue
        return float(max(0, val))

    # fallback
    val = mean_val
    if low is not None:
        val = max(val, low)
    if high is not None:
        val = min(val, high)
    return float(max(0, val))


In [ ]:

for pt in range(nPatients):
# for pt in range(3):

    fileName = matFiles[pt]

    ptID = os.path.splitext(fileName)[0]
    ptID = ptID.replace("_TDdataParamRecovery", "")

    print(f"processing pt {pt+1}/{nPatients}: {ptID}")

    matFile = os.path.join(inputFolderName, fileName)
    mat = loadmat(matFile, struct_as_record=False, squeeze_me=True)

    TDdataParamRecovery = mat["TDdataParamRecovery"]

    alphas = np.asarray(TDdataParamRecovery.a, dtype=float)
    bestAlphaPos = float(safe_scalar(TDdataParamRecovery.bestAlphaPos))
    bestAlphaNeg = float(safe_scalar(TDdataParamRecovery.bestAlphaNeg))

    bestAlphaPosIdx = np.where(np.isclose(alphas, bestAlphaPos))[0][0]
    bestAlphaNegIdx = np.where(np.isclose(alphas, bestAlphaNeg))[0][0]

    result = np.asarray(TDdataParamRecovery.result, dtype=str)
    reward = np.asarray(TDdataParamRecovery.Reward, dtype=float)
    isControl = np.asarray(TDdataParamRecovery.is_control).astype(int).squeeze()
    trial_type = np.asarray(TDdataParamRecovery.trial_type).astype(int).squeeze()

    bestRPE = np.asarray(TDdataParamRecovery.bestRewardPE, dtype=float).squeeze()
    best_expected_reward = np.asarray(TDdataParamRecovery.bestExpectedReward, dtype=float).squeeze()

    nTrials = len(result)

    resultSimulated = np.empty(nTrials, dtype=object)
    rewardSimulated = np.full(nTrials, np.nan)

    # scale for mapping more negative RPE to more pop probability
    neg_rpe_vals = np.abs(bestRPE[bestRPE < 0])
    neg_rpe_scale = np.nanmedian(neg_rpe_vals) if len(neg_rpe_vals) > 0 else 1.0
    if not np.isfinite(neg_rpe_scale) or neg_rpe_scale == 0:
        neg_rpe_scale = 1.0

    for trial in range(nTrials):

        # -----------------------------
        # control trials: keep identical
        # -----------------------------
        if isControl[trial] == 1:
            resultSimulated[trial] = result[trial]
            rewardSimulated[trial] = reward[trial]
            continue

        rpe = bestRPE[trial]
        this_result = result[trial]
        this_reward = reward[trial]
        this_trial_type = trial_type[trial]

        # safe fallback if unexpected trial type exists
        if this_trial_type not in reward_stats:
            resultSimulated[trial] = this_result
            rewardSimulated[trial] = this_reward
            continue

        mean_reward = reward_stats[this_trial_type]["mean"]
        std_reward = reward_stats[this_trial_type]["std"]

        # -----------------------------
        # RPE ~ 0 : keep same
        # -----------------------------
        if np.isclose(rpe, 0):
            resultSimulated[trial] = this_result
            rewardSimulated[trial] = this_reward

        # -----------------------------
        # positive RPE:
        # actual reward > expected reward
        # model would tend to stop earlier / more conservatively
        # -----------------------------
        elif rpe > 0:

            resultSimulated[trial] = "banked"

            if this_result == "banked":
                # simulate an earlier bank: reward <= participant reward
                simulated_reward = sample_truncated_normal(
                    mean_val=min(this_reward, mean_reward),
                    std_val=std_reward,
                    low=0,
                    high=this_reward
                )
            else:
                # participant popped, but model stops sooner -> bank something modest
                upper_bound = max(0, best_expected_reward[trial])
                simulated_reward = sample_truncated_normal(
                    mean_val=min(mean_reward, upper_bound),
                    std_val=std_reward,
                    low=0,
                    high=max(0, upper_bound)
                )

            rewardSimulated[trial] = simulated_reward

        # -----------------------------
        # negative RPE:
        # actual reward < expected reward
        # model would tend to keep pumping more
        # -----------------------------
        else:

            # if participant already popped, keep popped
            if this_result == "popped":
                resultSimulated[trial] = "popped"
                rewardSimulated[trial] = 0.0

            else:
                # participant banked, model goes further
                # simulate either:
                #   - larger banked reward
                #   - pop
                # pop probability increases with magnitude of negative RPE
                pop_prob = 1.0 - np.exp(-abs(rpe) / neg_rpe_scale)
                pop_prob = np.clip(pop_prob, 0.05, 0.95)

                if np.random.rand() < pop_prob:
                    resultSimulated[trial] = "popped"
                    rewardSimulated[trial] = 0.0
                else:
                    # simulate later bank: reward >= participant reward
                    simulated_reward = sample_truncated_normal(
                        mean_val=max(this_reward, mean_reward),
                        std_val=std_reward,
                        low=this_reward,
                        high=None
                    )
                    resultSimulated[trial] = "banked"
                    rewardSimulated[trial] = simulated_reward

    TDdataParamRecovery.resultSimulated = np.array(resultSimulated, dtype=object)
    TDdataParamRecovery.rewardSimulated = np.array(rewardSimulated, dtype=float)
    TDdataParamRecovery.nTrials = int(nTrials)

    # save to output folder
    outFile = os.path.join(outputFolderName, f"{ptID}_TDdataParamRecovery.mat")
    savemat(outFile, {"TDdataParamRecovery": TDdataParamRecovery}, do_compression=True)

    print(f"saved: {outFile}")


# debug

In [ ]:
fields = [f for f in dir(TDdataParamRecovery) if not f.startswith('_')]
print(fields)

In [ ]:
np.shape(isControl)